# Seizure and spike detection

Run seizure and spike detection
Activate spyglass environment first and install seizure_detection_ll functions

In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import spyglass.common as sgc
import spyglass.lfp as lfp
from spyglass.common.common_ephys import Raw
from seizure_detection_ll.utils import detect_global_epoch


### Key


In [ ]:
rat_name, date = "SC137", "20260218"
nwb_file_name = f"{rat_name}{date}_.nwb"
lfp_electrode_group_name = f"all_hippocampal_probes_all128ch_{rat_name}"
lfp_filter_name = "LFP 0-400 Hz"
probes_to_include = [3, 4, 5]
channels_per_probe = 128
n_detection_probes = 2  # For example use the first 2 probes for detection
out_dir = Path("/home/sc4712/src/repos/seizure_detection_ll/results") / f"{rat_name}{date}"


In [ ]:
sampling_rate = (Raw & {"nwb_file_name": nwb_file_name}).fetch1("sampling_rate")
decimation = round(sampling_rate / 1000)
target_sampling_rate = int(sampling_rate // decimation)
epochs = (sgc.IntervalList & {"nwb_file_name": nwb_file_name} &
          'interval_list_name REGEXP "^[0-9]{2}_[sr][0-9]$"').fetch("interval_list_name")

### LFP electrode group / LFPV1 (only if not already populated)

In [ ]:
electrodes = pd.DataFrame(sgc.Electrode & {"nwb_file_name": nwb_file_name}).sort_values("electrode_id")
electrode_ids = pd.concat([electrodes.iloc[(p-1)*128:p*128] for p in probes_to_include])["electrode_id"].tolist()

# lfp.lfp_electrode.LFPElectrodeGroup.create_lfp_electrode_group(
#     nwb_file_name=nwb_file_name, group_name=lfp_electrode_group_name,
#     electrode_list=electrode_ids, skip_duplicates=True)

# for epoch in epochs:
#     key = dict(nwb_file_name=nwb_file_name, lfp_electrode_group_name=lfp_electrode_group_name,
#                target_interval_list_name=epoch, filter_name=lfp_filter_name,
#                filter_sampling_rate=sampling_rate, target_sampling_rate=target_sampling_rate)
#     lfp.v1.LFPSelection.insert1(key, skip_duplicates=True)
#     lfp.v1.LFPV1().populate(key)


### Epoch detection

In [ ]:
seizure_params = dict(short_window_size_sec=1, long_window_size_sec=60, percent_overlap=50,
                      ratio_threshold=1.5, num_channels=32, duration_threshold_sec=10, max_gap_sec=1)
spike_params = dict(short_window_size_sec=0.05, long_window_size_sec=5, percent_overlap=50,
                    ratio_threshold=1.5, max_derivative_threshold=90000, amplitude_threshold=2000, num_channels=32)

lfp_entries = sorted((lfp.LFPOutput.LFPV1() & dict(nwb_file_name=nwb_file_name,
                        lfp_electrode_group_name=lfp_electrode_group_name,
                        filter_name=lfp_filter_name, filter_sampling_rate=sampling_rate)).fetch(as_dict=True),
                     key=lambda k: str(k.get("target_interval_list_name", "")))


In [ ]:
summary_rows, epoch_seizures, epoch_spikes = [], [], []
for key in lfp_entries:
    epoch = key.get("target_interval_list_name", str(key["merge_id"]))
    print(epoch)
    lfp_df = (lfp.LFPOutput & {"merge_id": key["merge_id"]}).fetch1_dataframe().iloc[:, :128*n_detection_probes]
    t = lfp_df.index.to_numpy(dtype=float)
    fs = 1 / np.median(np.diff(t)[np.diff(t) > 0])
    r = detect_global_epoch(lfp_df, fs, seizure_params, spike_params, channels_per_probe=128,
                            flat_fraction=0.90, flat_tolerance=0.01, min_flat_sec=0.25, pad_sec=0.5)
    seizures, spikes = r["seizure_intervals_df"].copy(), r["spike_intervals_df"].copy()
    seizures.insert(0, "epoch", epoch)
    spikes.insert(0, "epoch", epoch)
    epoch_seizures.append(seizures)
    epoch_spikes.append(spikes)
    summary_rows.append(dict(epoch=epoch, merge_id=key["merge_id"],
                             duration_min=r["duration_sec"]/60, n_seizures=len(seizures),
                             n_spikes=len(spikes), timestamp_gap_min=r["timestamp_gap_sec"]/60,
                             excluded_all_probes_min=r["excluded_all_probes_sec"]/60))
    del lfp_df, r
    gc.collect()

lfp_detection_summary_df = pd.DataFrame(summary_rows)
animal_seizures_df = pd.concat(epoch_seizures, ignore_index=True)
animal_spikes_df = pd.concat(epoch_spikes, ignore_index=True)
lfp_detection_summary_df[["epoch", "duration_min", "n_seizures", "n_spikes"]]


In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)
lfp_detection_summary_df.to_csv(out_dir / "detection_summary.csv", index=False)
animal_seizures_df.to_csv(out_dir / "seizures.csv", index=False)
animal_spikes_df.to_csv(out_dir / "spikes.csv", index=False)


### Optional: 75th-percentile amplitude and derivative by epoch

In [ ]:
threshold_rows = []
for key in lfp_entries:
    epoch = key.get("target_interval_list_name", str(key["merge_id"]))
    lfp_df = (lfp.LFPOutput & {"merge_id": key["merge_id"]}).fetch1_dataframe().iloc[:, :128*n_detection_probes:4]
    x, t = lfp_df.to_numpy(dtype=np.float32), lfp_df.index.to_numpy(dtype=float)
    idx = np.sort(np.random.default_rng(42).choice(len(t)-1, size=min(10000, len(t)-1), replace=False))
    dt = t[idx+1] - t[idx]
    valid = np.isfinite(dt) & (dt > 0)
    threshold_rows.append(dict(epoch=epoch, amplitude_75=np.nanpercentile(np.abs(x[idx]), 75),
                               derivative_75=np.nanpercentile(np.abs((x[idx[valid]+1] - x[idx[valid]]) / dt[valid, None]), 75)))
threshold_df = pd.DataFrame(threshold_rows)
threshold_df
